Sequential Chats and Customer Onboarding

## Setup

In [1]:
GEMINI_API_KEY = ""

llm_config = {
    "config_list": [
        {
            "model": "gemini-3.5-flash",
            "api_type": "google",
            "api_key": GEMINI_API_KEY
        }
    ],
    "timeout": 600,
    "cache_seed": 42
}

In [2]:
#%pip install -U google-generativeai vertexai jsonschema

In [3]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent(
    name="assistant",
    llm_config=llm_config
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    code_execution_config=False
)

C:\Users\manju\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\manju\anaconda3\lib\site-packages\autogen\oai\gemini.py:65: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
C:\Users\manju\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.20). Google will not post 

Patching name='__init__', member=<function BedrockClient.__init__ at 0x000001A8D4AA0310>, patched=<function function.__call__ at 0x000001A8D4AA04C0>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x000001A8D4AA0790>, patched=<function function.__call__ at 0x000001A8D4AA09D0>
Patching name='create', member=<function BedrockClient.create at 0x000001A8D4AA0700>, patched=<function function.__call__ at 0x000001A8D4AA0A60>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x000001A8D4AA0820>, patched=<function function.__call__ at 0x000001A8D4AA0AF0>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x000001A8D4AA0550>, patched=<function function.__call__ at 0x000001A8D4AA0B80>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x000001A8D4AA05E0>, patched=<function function.__call__ at 0x000001A8D4AA0C10>
Patching name='parse_params', 

C:\Users\manju\anaconda3\lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


Patching name='__init__', member=<function LLMLingua.__init__ at 0x000001A8ED128CA0>, patched=<function function.__call__ at 0x000001A8ED128C10>
Patching name='compress_text', member=<function LLMLingua.compress_text at 0x000001A8ED128D30>, patched=<function function.__call__ at 0x000001A8ED128EE0>


In [4]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent(
    name="assistant",
    llm_config=llm_config
)

In [5]:
llm_config = {"model": "llama3","api_type":"ollama"}

In [6]:
from autogen import ConversableAgent

## Creating the needed agents

In [8]:
# Use your Gemini API key directly
GEMINI_API_KEY = ""

llm_config = {
    "config_list": [
        {
            "model": "gemini-3.5-flash",
            "api_type": "google",
            "api_key": GEMINI_API_KEY
        }
    ],
    "timeout": 600,
    "cache_seed": 42
}

from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",
    system_message="""You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather the customer's name and location.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.""",
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",
    system_message="""You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather the customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.""",
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",
    system_message="""You are a helpful customer service agent.
    You are here to provide fun for the customer based on their
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.""",
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in (msg.get("content") or "").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in (msg.get("content") or "").lower(),
)

## Creating tasks

Now, you can craft a series of tasks to facilitate the onboarding process.

In [9]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

## Start the onboarding process

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [11]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)



********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  manjula, pollachi


customer_proxy_agent (to onboarding_personal_information_agent):

manjula, pollachi

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
{
  "name": "manjula",
  "location": ""
}
{"name": "Manjula", "location": "Pollachi"}

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  good but need of improvement


customer_proxy_agent (to onboarding_topic_preference_agent):

good but need of improvement

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
{
  "name": "manjula",
  "location": ""
}
The user asked about reading topic preferences, but no specific interests or details were exchanged.
{"name": "Manjula", "location": "Pollachi"}
The user asked Manjula, who is located in Pollachi, about her preferred reading topics, but no specific reading interests have been provided yet.

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Hello Manjula! Greetings to the lush, green "Coconut Capital" of Pollach

## Print out the summary

In [12]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

{'content': '{"name": "Manjula", "location": "Pollachi"}', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'The user asked Manjula, who is located in Pollachi, about her preferred reading topics, but no specific reading interests have been provided yet.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'Manjula from Pollachi is seeking engaging reading material, and the assistant offered tailored options across several fun categories—including animal facts, film trivia, unusual historical stories, and jokes—to help determine her reading preferences.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}




In [13]:
summary[0]

{'content': '{"name": "Manjula", "location": "Pollachi"}',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [14]:
#!pip install pandas

In [16]:
import ast
import json
import pandas as pd

# Get the assistant's response content
data_str = summary[0]["content"]

# Convert JSON string to a Python dictionary
data_dict = json.loads(data_str)

# Convert dictionary to DataFrame
df = pd.DataFrame([data_dict])

df

,name,location
0,Manjula,Pollachi


In [17]:
df["TopicPreference"]=summary[1]

In [18]:
df

,name,location,TopicPreference
0,Manjula,Pollachi,NaN


In [19]:
df["Enagement"]=summary[2]

In [20]:
df

,name,location,TopicPreference,Enagement
0,Manjula,Pollachi,NaN,NaN


In [21]:
!pip install gspread
!pip install oauth2client

In [22]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'websiteleadchatgptapi-a46f50ae1507.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1w-19wp0W2nRV5Pde5gim6kKlCoPTjv6mT3A8lAdNrcM'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


## Print out the cost

In [24]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 99, 'completion_tokens': 34, 'total_tokens': 133}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 99, 'completion_tokens': 34, 'total_tokens': 133}}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 178, 'completion_tokens': 47, 'total_tokens': 225}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 178, 'completion_tokens': 47, 'total_tokens': 225}}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 876, 'completion_tokens': 487, 'total_tokens': 1363}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 876, 'completion_tokens': 487, 'total_tokens': 1363}}}


